In [14]:
!which python

/home/user/jfayzullaev/stellar-clustering/.venv-vis/bin/python


In [15]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    normalized_mutual_info_score as NMI,
    adjusted_rand_score as ARI,
    adjusted_mutual_info_score as AMI,
    fowlkes_mallows_score as FMI,
    homogeneity_completeness_v_measure
)

In [16]:
LPA_COMM_TX_FILE   = "lpa_tx_lcc_lpa_communities.csv" 



In [17]:
NORM_LABELS_PATH = os.path.expanduser("~/stellar-clustering/publication/labeled-data/normalization/labels_mapped_normalized.csv") 

In [18]:
N_SPLITS = 5
RANDOM_STATE = 42

In [19]:
def overall_purity(df_comm_name: pd.DataFrame) -> float:
    if df_comm_name.empty:
        return np.nan
    counts = df_comm_name.groupby(['community', 'name']).size().reset_index(name='cnt')
    totals = counts.groupby('community')['cnt'].sum()
    max_per_comm = counts.groupby('community')['cnt'].max()
    return float(max_per_comm.sum() / totals.sum())

In [ ]:
def load_communities_fixed_resolution(lpa_file: str) -> pd.DataFrame:

    df = pd.read_csv(lpa_file)

    if 'account_id' not in df.columns and 'node' in df.columns:
        df = df.rename(columns={'node': 'account_id'})
    if 'community' not in df.columns:
        if 'lpa_comm' in df.columns:
            df = df.rename(columns={'lpa_comm': 'community'})
        elif 'comm' in df.columns:
            df = df.rename(columns={'comm': 'community'})

    df = df[['account_id', 'community']].dropna().drop_duplicates()

    try:
        df['account_id'] = df['account_id'].astype(int)
    except Exception:
        df['account_id'] = df['account_id'].astype(str)
    return df


In [ ]:
def evaluate_cv(
    labels_path: str,
    comm_df: pd.DataFrame,
    label_col: str = "name",
    n_splits: int = 5,
    random_state: int = 42
):


    labels = (pd.read_csv(labels_path)
                .dropna(subset=['account_id', label_col])
                .drop_duplicates(subset=['account_id'])
                .rename(columns={label_col: 'name'}))

    try:
        labels['account_id'] = labels['account_id'].astype(int)
        comm_cast = comm_df.copy()
        comm_cast['account_id'] = comm_cast['account_id'].astype(int)
    except Exception:
        labels['account_id'] = labels['account_id'].astype(str)
        comm_cast = comm_df.copy()
        comm_cast['account_id'] = comm_cast['account_id'].astype(str)

    joined = labels.merge(comm_cast, on='account_id', how='inner')

    n_labeled = len(labels)
    n_joined  = len(joined)
    coverage  = (n_joined / n_labeled) if n_labeled else 0.0
    
    le = LabelEncoder()
    y_all = le.fit_transform(joined['name'].values)
    X_ids = joined['account_id'].values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_ids, y_all), start=1):
        df_tr = joined.iloc[tr_idx].copy()
        df_te = joined.iloc[te_idx].copy()


        y_true_tr = le.transform(df_tr['name'])
        y_pred_tr = df_tr['community'].values
        
        nmi_tr = NMI(y_true_tr, y_pred_tr)
        ari_tr = ARI(y_true_tr, y_pred_tr)
        ami_tr = AMI(y_true_tr, y_pred_tr)
        fmi_tr = FMI(y_true_tr, y_pred_tr)
        homo_tr, comp_tr, v_tr = homogeneity_completeness_v_measure(y_true_tr, y_pred_tr)
        purity_tr = overall_purity(df_tr[['community', 'name']])


        y_true_te = le.transform(df_te['name'])
        y_pred_te = df_te['community'].values
        
        nmi_te = NMI(y_true_te, y_pred_te)
        ari_te = ARI(y_true_te, y_pred_te)
        ami_te = AMI(y_true_te, y_pred_te)
        fmi_te = FMI(y_true_te, y_pred_te)
        homo_te, comp_te, v_te = homogeneity_completeness_v_measure(y_true_te, y_pred_te)
        purity_te = overall_purity(df_te[['community', 'name']])

        rows.append({
            'fold': fold,
            'n_train': len(df_tr),
            'n_test': len(df_te),
            'train_frac': len(df_tr) / len(joined),

            # Train metrics
            'NMI_train': nmi_tr,
            'ARI_train': ari_tr,
            'AMI_train': ami_tr,
            'FMI_train': fmi_tr,
            'Homogeneity_train': homo_tr,
            'Completeness_train': comp_tr,
            'V-measure_train': v_tr,
            'Purity_train': purity_tr,

            # Test metrics
            'NMI_test': nmi_te,
            'ARI_test': ari_te,
            'AMI_test': ami_te,
            'FMI_test': fmi_te,
            'Homogeneity_test': homo_te,
            'Completeness_test': comp_te,
            'V-measure_test': v_te,
            'Purity_test': purity_te,
        })

    per_fold_df = pd.DataFrame(rows)


    metric_names = ['NMI', 'ARI', 'AMI', 'FMI', 'Homogeneity', 'Completeness', 'V-measure', 'Purity']
    averages = {}
    
    for metric in metric_names:
        averages[f'Avg_{metric}_train'] = float(np.nanmean(per_fold_df[f'{metric}_train'].values))
        averages[f'Avg_{metric}_test'] = float(np.nanmean(per_fold_df[f'{metric}_test'].values))
        averages[f'Std_{metric}_test'] = float(np.nanstd(per_fold_df[f'{metric}_test'].values))

    averages['Avg_train_frac'] = float(np.nanmean(per_fold_df['train_frac'].values))

    coverage_info = {
        'n_labeled': int(n_labeled),
        'n_joined': int(n_joined),
        'coverage': float(coverage),
    }

    return per_fold_df, averages, coverage_info

# Transactions 

In [ ]:
comm_df = load_communities_fixed_resolution(LPA_COMM_TX_FILE)

print(f"Rows: {len(comm_df):,},  Unique accounts: {comm_df['account_id'].nunique():,}")

Loaded LPA communities from: lpa_tx_lcc_lpa_communities.csv
Rows: 4,315,652  |  Unique accounts: 4,315,652


In [23]:
norm_per_fold, norm_avg, norm_cov = evaluate_cv(
    labels_path=NORM_LABELS_PATH,
    comm_df=comm_df,
    label_col="name",
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE
)

print("NORMALIZED labels: per-fold metrics (TEST)")
display(norm_per_fold)

print("\nNORMALIZED labels: averages (TEST)")
for k, v in norm_avg.items():
    print(f"{k}: {v:.6f}")
print(f"Coverage: {norm_cov['coverage']:.2%}  ({norm_cov['n_joined']}/{norm_cov['n_labeled']})")


NORMALIZED labels: per-fold metrics (TEST)


/home/user/jfayzullaev/stellar-clustering/.venv-vis/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


,fold,n_train,n_test,train_frac,NMI_train,ARI_train,AMI_train,FMI_train,Homogeneity_train,Completeness_train,V-measure_train,Purity_train,NMI_test,ARI_test,AMI_test,FMI_test,Homogeneity_test,Completeness_test,V-measure_test,Purity_test
0,1,6668,1668,0.799904,0.112813,-0.001237,0.061339,0.359441,0.333580,0.067885,0.112813,0.939262,0.123299,-0.000069,0.056122,0.350249,0.386144,0.073362,0.123299,0.941247
1,2,6669,1667,0.800024,0.112729,-0.001884,0.060843,0.353575,0.333620,0.067823,0.112729,0.939721,0.126975,0.003383,0.062209,0.374705,0.396660,0.075586,0.126975,0.940012
2,3,6669,1667,0.800024,0.114394,-0.000641,0.063270,0.358964,0.339730,0.068776,0.114394,0.939571,0.117788,-0.002312,0.053316,0.352169,0.367234,0.070143,0.117788,0.940012
3,4,6669,1667,0.800024,0.114204,-0.000842,0.063122,0.360334,0.338617,0.068684,0.114204,0.939721,0.118194,-0.001502,0.052058,0.346731,0.370799,0.070302,0.118194,0.938812
4,5,6669,1667,0.800024,0.113431,-0.000355,0.063034,0.355531,0.337345,0.068177,0.113431,0.939421,0.121995,-0.004057,0.053885,0.365593,0.377985,0.072735,0.121995,0.940612



NORMALIZED labels: averages (TEST)
Avg_NMI_train: 0.113514
Avg_NMI_test: 0.121650
Std_NMI_test: 0.003407
Avg_ARI_train: -0.000992
Avg_ARI_test: -0.000911
Std_ARI_test: 0.002504
Avg_AMI_train: 0.062322
Avg_AMI_test: 0.055518
Std_AMI_test: 0.003595
Avg_FMI_train: 0.357569
Avg_FMI_test: 0.357889
Std_FMI_test: 0.010561
Avg_Homogeneity_train: 0.336579
Avg_Homogeneity_test: 0.379764
Std_Homogeneity_test: 0.010646
Avg_Completeness_train: 0.068269
Avg_Completeness_test: 0.072426
Std_Completeness_test: 0.002034
Avg_V-measure_train: 0.113514
Avg_V-measure_test: 0.121650
Std_V-measure_test: 0.003407
Avg_Purity_train: 0.939539
Avg_Purity_test: 0.940139
Std_Purity_test: 0.000805
Avg_train_frac: 0.800000
Coverage: 100.00%  (8336/8336)


In [24]:
summary_metrics = ['NMI', 'ARI', 'AMI', 'FMI', 'Homogeneity', 'Completeness', 'V-measure', 'Purity']
summary_data = []

for metric in summary_metrics:
    summary_data.append({
        'Metric': metric,
        'Mean': norm_avg[f'Avg_{metric}_test'],
        'Std': norm_avg[f'Std_{metric}_test']
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f"\nCoverage: {norm_cov['coverage']:.2%}  ({norm_cov['n_joined']}/{norm_cov['n_labeled']})")


,Metric,Mean,Std
0,NMI,0.121650,0.003407
1,ARI,-0.000911,0.002504
2,AMI,0.055518,0.003595
3,FMI,0.357889,0.010561
4,Homogeneity,0.379764,0.010646
5,Completeness,0.072426,0.002034
6,V-measure,0.121650,0.003407
7,Purity,0.940139,0.000805



Coverage: 100.00%  (8336/8336)


In [25]:
TX_PATH = 'cross-validation'
os.makedirs(TX_PATH, exist_ok=True) 

norm_per_fold.to_csv(f"{TX_PATH}/lpa_cv_results_norm_per_fold.csv", index=False)
pd.DataFrame([{**norm_avg, **norm_cov}]).to_csv(f"{TX_PATH}/lpa_cv_results_norm_summary.csv", index=False)
summary_df.to_csv(f"{TX_PATH}/lpa_cv_results_norm_summary_table.csv", index=False)